# FastSpeech 2: English Text-to-Speech Training & Evaluation
### Course Assignment: FastSpeech 2-Based TTS (Svarah English Dataset)

This notebook runs the complete end-to-end pipeline:
1. Environment configuration and dependency verification.
2. Acoustic feature extraction (HiFi-GAN compatible Log-Mel, Continuous F0, STFT Energy).
3. Genuine phoneme-level forced alignment (MMS-FA / Acoustic Segmenter).
4. FastSpeech 2 training with fully-masked multi-task loss.
5. Synthesis of the 5 benchmark sentences (E1–E4) with HiFi-GAN neural vocoding and variance plots.

## 1. Setup Environment & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q g2p_en pyworld librosa soundfile matplotlib pyyaml tqdm torch torchaudio soxr
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('cmudict', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Dependencies installed successfully.')

## 2. Unpack Project Code & Locate Dataset

In [ ]:
import os, glob, zipfile, shutil

os.makedirs('/kaggle/working/project', exist_ok=True)

# Locate uploaded project code zip or copy from /kaggle/input
src_zips = glob.glob('/kaggle/input/**/FastSpeech2_Project_Source.zip', recursive=True) + \
           glob.glob('/kaggle/input/**/fastspeech2*.zip', recursive=True)
if src_zips:
    with zipfile.ZipFile(src_zips[0], 'r') as z:
        z.extractall('/kaggle/working/project')
    print('Extracted project from:', src_zips[0])
else:
    for d in glob.glob('/kaggle/input/*'):
        if os.path.isdir(d) and not 'dataset' in d.lower():
            shutil.copytree(d, '/kaggle/working/project', dirs_exist_ok=True)
            print('Copied project from:', d)

# Locate English dataset zip
dataset_zips = glob.glob('/kaggle/input/**/english*.zip', recursive=True)
if dataset_zips:
    data_dest = '/kaggle/working/project/English/data/Svarah'
    os.makedirs(data_dest, exist_ok=True)
    with zipfile.ZipFile(dataset_zips[0], 'r') as z:
        z.extractall(data_dest)
    print('Extracted English dataset from:', dataset_zips[0])

# Create results directories
os.makedirs('/kaggle/working/project/English/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/test_generated_audios', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/variance_adaptor_plots/duration', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/variance_adaptor_plots/pitch', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/variance_adaptor_plots/energy', exist_ok=True)
print('Working project directory ready.')

## 3. Run Preprocessing & Forced Alignment Pipeline

In [ ]:
%cd /kaggle/working/project/English
!python preprocess.py config.yaml

## 4. Train English FastSpeech 2 Model

In [ ]:
%cd /kaggle/working/project/English
!python train.py config.yaml

## 5. Run Inference & Vocoding with HiFi-GAN

In [ ]:
%cd /kaggle/working/project/English
!python inference.py --config config.yaml --checkpoint checkpoints/best_model.pth

## 6. Test Arbitrary Text CLI Synthesis

In [ ]:
%cd /kaggle/working/project/English
!python inference.py --text "Hello, how are you today? FastSpeech 2 synthesis is now completely verified!" --output /kaggle/working/custom_sample.wav
import IPython.display as ipd
ipd.Audio('/kaggle/working/custom_sample.wav')

## 7. Package Deliverables for Download

In [ ]:
%cd /kaggle/working
!zip -r english_results.zip project/Results/English project/English/checkpoints
print('english_results.zip is ready for download in Kaggle Output!')